# Lab 04: Linear Scores and Multiclass SVM Loss



By the end of this lab you will compute linear class scores, implement the
loop-based multiclass SVM loss and gradient, and verify the gradient with
finite differences.



## 1. Setup

Select a Python environment with the packages in the repository root
`requirements.txt`. Run this notebook from `LAB04` or the repository root.

The lab runs on CPU. The setup imports the local `linear_classifier.py`
and shared gradient-check helper. After editing the implementation, rerun
the validation cells; autoreload refreshes the module.


In [1]:
from pathlib import Path
import sys

lab_dir = Path.cwd()
if not (lab_dir / "linear_classifier.py").is_file():
    lab_dir = lab_dir / "LAB04"
repo_root = lab_dir.parent
if not (repo_root / "dlcv2026").is_dir():
    raise FileNotFoundError("Open this notebook from the cloned repository with its dlcv2026 folder.")
sys.path[:0] = [str(lab_dir), str(repo_root)]

%load_ext autoreload
%autoreload 2

from pathlib import Path
import torch

import dlcv2026
import linear_classifier

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

print("PyTorch:", torch.__version__)
print("Implementation:", Path(linear_classifier.__file__).resolve())


PyTorch: 2.14.0+cpu
Implementation: /workspace/scratch/fc683240ee0e/otherlabs-check/fixed/LAB04/linear_classifier.py


## 2. Linear scores

For a minibatch, a linear classifier computes $S=XW+b$. Each row of $S$
contains one score per class. Run the fixture and identify the predicted class
for each example before looking at the printed result.


In [2]:
X_demo = torch.tensor([[1.0, 2.0], [-1.0, 1.0]])
W_demo = torch.tensor([[0.5, -1.0, 0.0], [1.0, 0.5, -0.5]])
b_demo = torch.tensor([0.1, 0.0, 0.2])

scores_demo = linear_classifier.linear_scores(X_demo, W_demo, b_demo)
predictions_demo = scores_demo.argmax(dim=1)
print("scores:\n", scores_demo)
print("predicted classes:", predictions_demo.tolist())

assert scores_demo.shape == (2, 3)
assert predictions_demo.dtype == torch.int64


scores:
 tensor([[ 2.6000,  0.0000, -0.8000],
        [ 0.6000,  1.5000, -0.3000]])
predicted classes: [0, 1]


## 3. Multiclass SVM loss and gradient

For example $i$, correct class $y_i$, and margin $\Delta=1$:

$$
L_i = \sum_{j \neq y_i} \max(0, s_j - s_{y_i} + \Delta).
$$

Open `linear_classifier.py` and complete `svm_loss_naive`. Use explicit
loops. When a margin is positive, update both the loss and the two affected
columns of the gradient. Complete all four TODOs: accumulate positive
margins, update the two gradient columns, average over the minibatch, and
return `loss, dW`.

For Lab 04, use the unregularized objective:

$$
L = \frac{1}{N}\sum_i L_i.
$$

Keep `reg=0.0` throughout this lab. Lecture 3B and Lab 05 introduce L2
regularization.


In [3]:
# Save linear_classifier.py, then rerun this cell.
X = torch.tensor([[1.0, 0.0], [0.0, 1.0], [-1.0, -1.0]])
W = torch.tensor([[1.0, -1.0, 0.0], [0.0, 1.0, -1.0]])
y = torch.tensor([2, 0, 1], dtype=torch.int64)

loss, gradient = linear_classifier.svm_loss_naive(
    W.clone(), X, y, reg=0.0
)
print("unregularized loss:", float(loss))
print("gradient:\n", gradient)

assert loss.ndim == 0
assert gradient.shape == W.shape
assert torch.allclose(loss, torch.tensor(2.0), atol=1e-12)


unregularized loss: 2.0
gradient:
 tensor([[ 0.3333,  0.3333, -0.6667],
        [-0.3333,  0.6667, -0.3333]])


## 4. Numerical gradient check

Finite differences provide an independent check of your analytic gradient.
Most relative errors should be below $10^{-5}$. A few larger values can occur
exactly at the nondifferentiable hinge boundary; the random fixture avoids that
case in normal runs.


In [4]:
help(dlcv2026.grad_check_sparse)

torch.manual_seed(4)
X_check = torch.randn(7, 5)
W_check = 0.1 * torch.randn(5, 4)
y_check = torch.tensor([0, 1, 2, 3, 0, 1, 2], dtype=torch.int64)

loss_check, grad_check = linear_classifier.svm_loss_naive(
    W_check, X_check, y_check, reg=0.0
)
f = lambda weights: linear_classifier.svm_loss_naive(
    weights, X_check, y_check, reg=0.0
)[0]
errors = dlcv2026.grad_check_sparse(
    f, W_check, grad_check, num_checks=10, h=1e-6
)
print("maximum relative error:", max(errors))
assert max(errors) < 1e-5


Help on function grad_check_sparse in module dlcv2026.gradient:

grad_check_sparse(f, x, analytic_grad, num_checks=10, h=1e-05)
    Check random coordinates with centered finite differences.

    Return relative errors, restoring each perturbed coordinate even if the
    objective raises an exception. The input and gradient shapes must match.

numerical: -0.358784, analytic: -0.358784, relative error: 1.56e-10
numerical: 0.925584, analytic: 0.925584, relative error: 1.21e-10
numerical: -0.006152, analytic: -0.006152, relative error: 2.18e-09
numerical: 0.655455, analytic: 0.655455, relative error: 2.52e-10
numerical: -1.264560, analytic: -1.264560, relative error: 2.25e-10
numerical: -0.202469, analytic: -0.202469, relative error: 2.68e-10
numerical: -1.492622, analytic: -1.492622, relative error: 1.24e-10
numerical: 0.821755, analytic: 0.821755, relative error: 3.14e-11
numerical: 0.843319, analytic: 0.843319, relative error: 9.15e-11
numerical: 0.925584, analytic: 0.925584, relative 


1. If the correct class score increase, the loss for this example will stay same or go down, because the margins become smaller.
2. One margin changes two columns because it compare the wrong class with the correct one. We decrease the wrong class j and increase the correct class y_i, so both are needed.
3. If delta is bigger, more margins can become positive, so more classes will contribute to the loss. This force the model to have a bigger gap between the correct class and the wrong classes.